In [2]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 1: Setup & Configuration
# ============================================================
# Part B of the Digikala project: cleaning, splitting, masking,
# and production of standard artifacts.
#
# This cell does NOT read, clean, or write any data.
# It only:
#   - imports libraries
#   - defines shared paths and constants
#   - verifies that input files and required packages are present
# ============================================================

from __future__ import annotations

# --- Standard library --------------------------------------------------------
import hashlib   # for the deterministic masking protocol (B4)
import json      # for split_manifest.json (B5)
import random    # for any bounded sampling, always with a seed
from pathlib import Path

# --- Third-party: data processing --------------------------------------------
import polars as pl

# --- Third-party: Persian NLP ------------------------------------------------
import hazm      # normalizer for Persian text (B1)

# --- Reproducibility seeds (fixed by the Definition, must not change) ------
SPLIT_SEED = 2026
MASK_SEED = "digikala-semi-supervised-v1"
MASK_RATE = 0.20

# --- Project layout ----------------------------------------------------------
PROJECT_ROOT = Path("/home/jvd/Desktop/persian-nlp-pipeline")
RAW_DIR       = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR    = PROJECT_ROOT / "data" / "splits"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"

COMMENTS_CSV = RAW_DIR / "digikala-comments.csv"
PRODUCTS_CSV = RAW_DIR / "digikala-products.csv"

# --- Output files that Part B will produce -----------------------------------
# (declared here as a single source of truth, populated in later cells)
COMMENTS_CLEAN_PARQUET = PROCESSED_DIR / "comments_clean.parquet"
TRAIN_SUPERVISED_CSV   = SPLITS_DIR / "train_supervised.csv"
TRAIN_PSEUDO_UNLABELED_CSV = SPLITS_DIR / "train_pseudo_unlabeled.csv"
VALIDATION_CSV         = SPLITS_DIR / "validation.csv"
TEST_CSV               = SPLITS_DIR / "test.csv"
SPLIT_MANIFEST_JSON    = SPLITS_DIR / "split_manifest.json"

# --- Create output directories if they do not exist --------------------------
for _d in (PROCESSED_DIR, SPLITS_DIR, ARTIFACTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# --- Fail-fast: input files must exist --------------------------------------
for _p in (COMMENTS_CSV, PRODUCTS_CSV):
    assert _p.exists(), f"Missing input file: {_p}"

# --- Fail-fast: hazm must import (installed in the uv environment) ----------
# The version string is handy when the normalizer's behavior changes across
# hazm releases.
_hazm_version = getattr(hazm, "__version__", "unknown")

# --- Ready message -----------------------------------------------------------
print("Part B environment ready.")
print(f"PROJECT_ROOT   : {PROJECT_ROOT}")
print(f"COMMENTS_CSV   : {COMMENTS_CSV}  "
      f"({COMMENTS_CSV.stat().st_size / 1024**2:,.1f} MB)")
print(f"PRODUCTS_CSV   : {PRODUCTS_CSV}  "
      f"({PRODUCTS_CSV.stat().st_size / 1024**2:,.1f} MB)")
print(f"PROCESSED_DIR  : {PROCESSED_DIR}")
print(f"SPLITS_DIR     : {SPLITS_DIR}")
print(f"ARTIFACTS_DIR  : {ARTIFACTS_DIR}")
print()
print(f"SPLIT_SEED     : {SPLIT_SEED}")
print(f"MASK_SEED      : {MASK_SEED!r}")
print(f"MASK_RATE      : {MASK_RATE}")
print()
print(f"polars version : {pl.__version__}")
print(f"hazm version   : {_hazm_version}")
print()
print("Planned outputs (will be written by later cells):")
for p in (
    COMMENTS_CLEAN_PARQUET,
    TRAIN_SUPERVISED_CSV,
    TRAIN_PSEUDO_UNLABELED_CSV,
    VALIDATION_CSV,
    TEST_CSV,
    SPLIT_MANIFEST_JSON,
):
    print(f"  {p.relative_to(PROJECT_ROOT)}")

Part B environment ready.
PROJECT_ROOT   : /home/jvd/Desktop/persian-nlp-pipeline
COMMENTS_CSV   : /home/jvd/Desktop/persian-nlp-pipeline/data/raw/digikala-comments.csv  (1,219.3 MB)
PRODUCTS_CSV   : /home/jvd/Desktop/persian-nlp-pipeline/data/raw/digikala-products.csv  (246.7 MB)
PROCESSED_DIR  : /home/jvd/Desktop/persian-nlp-pipeline/data/processed
SPLITS_DIR     : /home/jvd/Desktop/persian-nlp-pipeline/data/splits
ARTIFACTS_DIR  : /home/jvd/Desktop/persian-nlp-pipeline/artifacts

SPLIT_SEED     : 2026
MASK_SEED      : 'digikala-semi-supervised-v1'
MASK_RATE      : 0.2

polars version : 2.1.0
hazm version   : unknown

Planned outputs (will be written by later cells):
  data/processed/comments_clean.parquet
  data/splits/train_supervised.csv
  data/splits/train_pseudo_unlabeled.csv
  data/splits/validation.csv
  data/splits/test.csv
  data/splits/split_manifest.json


In [3]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 2: B1 Persian Normalization
# ============================================================
# Goal:
#   1. Collect evidence for normalization decisions on a 5,000-row
#      deterministic sample (seed = SPLIT_SEED).
#   2. Build a testable normalize_persian() function using hazm.Normalizer,
#      with URL/HTML stripping done BEFORE hazm, half-space preserved,
#      negation preserved, and emojis kept.
#   3. Sanity-test the function on a handful of tricky sentences.
#
# No files are written. Raw CSVs remain untouched.
# ============================================================

import re

# --- 1. Lazy load the raw body column (only body, only what we need) ------
# We deliberately scan only the `body` column. This keeps the sample cheap.
comments_body_lf = pl.scan_csv(
    COMMENTS_CSV,
    separator=",",
    encoding="utf8",
    ignore_errors=False,
).select("body")

# --- 2. Deterministic sample of 5,000 rows --------------------------------
SAMPLE_N = 5_000
sample_bodies = (
    comments_body_lf
    .filter(pl.col("body").is_not_null())
    .collect(engine="streaming")
    .sample(n=SAMPLE_N, seed=SPLIT_SEED, with_replacement=False)
)
print(f"Sample size: {sample_bodies.height:,} rows (seed = {SPLIT_SEED})")
print()

# --- 3. Evidence: how prevalent are each normalization-relevant issue? ----
# Every check is a Polars expression over `body`, evaluated on the sample.
# We report absolute counts and percentages.
N = sample_bodies.height

def pct(count: int) -> str:
    return f"{100 * count / N:6.2f}%"

evidence = sample_bodies.select(
    pl.len().alias("n"),
    pl.col("body").str.contains(r"[\u064A]").sum().alias("arabic_yeh"),
    pl.col("body").str.contains(r"[\u0643]").sum().alias("arabic_kaf"),
    pl.col("body").str.contains(r"\u200c").sum().alias("half_space"),
    pl.col("body").str.contains(r"https?://|www\.").sum().alias("url"),
    pl.col("body").str.contains(r"<[a-zA-Z/][^>]*>").sum().alias("html_tag"),
    pl.col("body").str.contains(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]").sum().alias("emoji"),
    pl.col("body").str.contains(r"[0-9]").sum().alias("ascii_digit"),
    pl.col("body").str.contains(r"[\u06F0-\u06F9]").sum().alias("persian_digit"),
    pl.col("body").str.contains(r"[\u0660-\u0669]").sum().alias("arabic_digit"),
    pl.col("body").str.contains(r"[\u200B-\u200F\u202A-\u202E\uFEFF]").sum().alias("zero_width_or_bidi"),
    pl.col("body").str.contains(r"\n").sum().alias("contains_newline"),
).row(0, named=True)

print(f"Normalization-relevant patterns in the sample (n={N}):")
for key, value in evidence.items():
    if key == "n":
        continue
    print(f"  {key:<18}  {value:>6,}  ({pct(value)})")
print()

# --- 4. Evidence: does half-space matter for negation? -------------------
# These are hand-picked patterns where removing the half-space changes
# meaning or readability. If they appear frequently, preserving half-space
# is empirically justified.
negation_patterns = [
    "نمی‌خوام",  # colloquial "I don't want" with half-space
    "نمی‌خواهم", # formal "I don't want" with half-space
    "نمی‌تونم",  # colloquial "I can't"
    "نمی‌شه",    # colloquial "it isn't possible"
    "نمی‌دونم",  # colloquial "I don't know"
]
negation_hits = sample_bodies.select(
    [pl.col("body").str.contains(p, literal=True).sum().alias(p) for p in negation_patterns]
).row(0, named=True)

print("Negation patterns with half-space (literal substring match):")
for pat, cnt in negation_hits.items():
    print(f"  {pat:<14}  {cnt:>5}  ({pct(cnt)})")
print()

# --- 5. Build the normalizer (single instance, reused for every row) -----
# hazm.Normalizer() preserves half-space. We wrap it rather than replace it.
_normalizer = hazm.Normalizer()

# Pre-compile regexes once; reused across millions of calls.
_URL_RE   = re.compile(r"https?://\S+|www\.\S+")
_HTML_RE  = re.compile(r"<[^>]+>")
_ZERO_RE  = re.compile(r"[\u200B-\u200D\uFEFF]")   # ZWSP/ZWJ/ZWNJ removal left to hazm
_WS_RE    = re.compile(r"[ \t\r\f\v]+")             # collapse only horizontal whitespace

def normalize_persian(text: str) -> str:
    """
    Normalize a Persian review body.

    Order of operations:
      1. Strip URLs and HTML tags (so hazm doesn't see them).
      2. Remove zero-width non-joiner-adjacent control chars (but NOT ZWNJ itself).
      3. Run hazm.Normalizer (handles Arabic yeh/kaf, digits, whitespace, half-space).
      4. Collapse runs of horizontal whitespace to a single space.
      5. Strip leading/trailing whitespace.

    Preserved on purpose:
      - Half-space (ZWNJ, U+200C) — critical for Persian meaning (negation, compounds).
      - Emojis — kept as potential sentiment signal.
      - Digits (Persian, Arabic, ASCII) — hazm converts them to ASCII, which we keep.
      - Punctuation — kept for now; will be handled during tokenization.

    Notes:
      - This function is idempotent for already-normalized text.
      - Empty input returns an empty string.
    """
    if not text:
        return ""

    # 1. Strip URLs and HTML before hazm.
    text = _URL_RE.sub(" ", text)
    text = _HTML_RE.sub(" ", text)

    # 2. Remove zero-width controls OTHER than ZWNJ (U+200C), which hazm keeps.
    #    ZWNJ = \u200C; we exclude it from removal.
    text = re.sub(r"[\u200B\u200D\uFEFF]", "", text)

    # 3. hazm.Normalizer: Arabic yeh/kaf, digits, half-space, whitespace.
    text = _normalizer.normalize(text)

    # 4. Collapse horizontal whitespace runs (but keep newlines as single \n).
    text = _WS_RE.sub(" ", text)

    # 5. Trim.
    return text.strip()

# --- 6. Sanity-test on tricky sentences ----------------------------------
test_cases = [
    "كتاب خوبي بود",                        # Arabic yeh + Arabic kaf
    "نمی‌خوام دیگه بخرم",                    # half-space + negation
    "قیمتش ۱۲۵۰۰۰ تومان بود",                # Persian digits
    "خیلی خوب بود 😍👌",                    # emojis
    "به http://example.com سر بزنید",        # URL
    "<b>عالی</b> بود",                        # HTML
    "خوب  بود\n\n  ولی گرون بود",           # multiple spaces + newlines
    "",                                       # empty
    "   ",                                    # whitespace only
]

print("Sanity checks — normalize_persian():")
for raw in test_cases:
    norm = normalize_persian(raw)
    print(f"  raw    : {raw!r}")
    print(f"  normal : {norm!r}")
    print()

# --- 7. Idempotence check -------------------------------------------------
# Running the normalizer twice must yield the same output as running once.
non_idempotent = 0
for raw in test_cases:
    once  = normalize_persian(raw)
    twice = normalize_persian(once)
    if once != twice:
        non_idempotent += 1
        print(f"NON-IDEMPOTENT: {raw!r}\n  once  = {once!r}\n  twice = {twice!r}")
print(f"Idempotence violations: {non_idempotent} / {len(test_cases)}")


Sample size: 5,000 rows (seed = 2026)

Normalization-relevant patterns in the sample (n=5000):
  arabic_yeh              49  (  0.98%)
  arabic_kaf              21  (  0.42%)
  half_space             150  (  3.00%)
  url                      0  (  0.00%)
  html_tag                 0  (  0.00%)
  emoji                   38  (  0.76%)
  ascii_digit             86  (  1.72%)
  persian_digit          176  (  3.52%)
  arabic_digit             2  (  0.04%)
  zero_width_or_bidi     151  (  3.02%)
  contains_newline       509  ( 10.18%)

Negation patterns with half-space (literal substring match):
  نمی‌خوام            0  (  0.00%)
  نمی‌خواهم           0  (  0.00%)
  نمی‌تونم            1  (  0.02%)
  نمی‌شه              0  (  0.00%)
  نمی‌دونم            1  (  0.02%)

Sanity checks — normalize_persian():
  raw    : 'كتاب خوبي بود'
  normal : 'کتاب خوبی بود'

  raw    : 'نمی\u200cخوام دیگه بخرم'
  normal : 'نمی\u200cخوام دیگه بخرم'

  raw    : 'قیمتش ۱۲۵۰۰۰ تومان بود'
  normal : 'قیمتش ۱۲۵۰۰۰

In [4]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 3: B2 Data Cleaning
# ============================================================
# Goal:
#   1. Convert sentinel strings ("nan", "NaN", "null", "None", "")
#      to real nulls in string columns.
#   2. Normalize body with normalize_persian() from B1,
#      plus explicit Persian→ASCII digit conversion.
#   3. Derive rating_band from rate (thresholds 2.5 and 4.0),
#      with an explicit is_valid_rating_band flag for rate outliers.
#   4. Remove duplicate IDs (exact + divergent).
#   5. Write data/processed/comments_clean.parquet.
#
# Writes exactly ONE file: data/processed/comments_clean.parquet.
# Raw CSVs remain untouched.
# ============================================================

import re

# --- 1. Persian → ASCII digit conversion (added on top of hazm) ---------
# hazm 0.12.1 does not convert Persian/Arabic digits by default.
# We convert them explicitly so the tokenizer sees a single digit surface form.
_DIGIT_MAP = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹"   # Persian digits
    "٠١٢٣٤٥٦٧٨٩",  # Arabic-Indic digits
    "01234567890123456789",
)

def normalize_persian_v2(text: str) -> str:
    """B2 version: B1 normalizer + Persian/Arabic digits → ASCII digits."""
    text = normalize_persian(text)          # from B1 cell
    return text.translate(_DIGIT_MAP)

# --- 2. Load only the columns we keep in the parquet ---------------------
KEEP_COLS = [
    "id", "body", "recommendation_status",
    "rate", "product_id", "is_buyer",
]
comments_lf = pl.scan_csv(
    COMMENTS_CSV,
    separator=",",
    encoding="utf8",
    ignore_errors=False,
).select(KEEP_COLS)

# --- 3. Convert sentinels to real nulls ---------------------------------
SENTINELS = ["nan", "NaN", "null", "None", ""]
comments_lf = comments_lf.with_columns(
    pl.when(pl.col("recommendation_status").is_in(SENTINELS))
      .then(pl.lit(None, dtype=pl.String))
      .otherwise(pl.col("recommendation_status"))
      .alias("recommendation_status"),
    pl.when(pl.col("body").is_in(SENTINELS))
      .then(pl.lit(None, dtype=pl.String))
      .otherwise(pl.col("body"))
      .alias("body"),
)

# --- 4. Drop duplicate IDs (exact + divergent) --------------------------
# Rule (documented): any id that appears more than once is removed entirely.
# This handles both exact duplicates and divergent duplicates uniformly.
# Counts are logged below for split_manifest.json.
n_before_dedup_lf = comments_lf.select(pl.len())

duplicate_ids_lf = (
    comments_lf
    .group_by("id")
    .agg(pl.len().alias("n"))
    .filter(pl.col("n") > 1)
    .select("id")
)

comments_dedup_lf = comments_lf.join(
    duplicate_ids_lf, on="id", how="anti"
)

n_after_dedup_lf = comments_dedup_lf.select(pl.len())

# --- 5. Derive rating_band and is_valid_rating_band --------------------
# rating_band values: low, medium, high, invalid (rate missing or out of [0,5])
# is_valid_rating_band: True iff 0 <= rate <= 5
comments_dedup_lf = comments_dedup_lf.with_columns(
    pl.when(pl.col("rate").is_null())
      .then(pl.lit("invalid", dtype=pl.String))
      .when((pl.col("rate") < 0) | (pl.col("rate") > 5))
      .then(pl.lit("invalid", dtype=pl.String))
      .when(pl.col("rate") < 2.5)
      .then(pl.lit("low", dtype=pl.String))
      .when(pl.col("rate") < 4.0)
      .then(pl.lit("medium", dtype=pl.String))
      .otherwise(pl.lit("high", dtype=pl.String))
      .alias("rating_band"),
    ((pl.col("rate").is_not_null())
     & (pl.col("rate") >= 0)
     & (pl.col("rate") <= 5))
      .alias("is_valid_rating_band"),
)

# --- 6. Normalize body (B1 + digit conversion) -------------------------
# map_batches: apply a Python function to the whole Series at once.
# This is faster than map_elements for row-wise string transforms.
comments_dedup_lf = comments_dedup_lf.with_columns(
    pl.col("body")
      .map_batches(
          lambda s: s.map_elements(
              normalize_persian_v2,
              return_dtype=pl.String,
          ),
          return_dtype=pl.String,
      )
      .alias("body_normalized"),
).drop("body")

# --- 7. Materialize and log -------------------------------------------------
# collect(engine="streaming") keeps peak memory bounded.
comments_clean = comments_dedup_lf.collect(engine="streaming")

n_before_dedup = n_before_dedup_lf.collect().item()
n_after_dedup  = n_after_dedup_lf.collect().item()
n_rows_clean   = comments_clean.height

print("B2 Data Cleaning summary:")
print(f"  rows before dedup        : {n_before_dedup:>10,}")
print(f"  rows after dedup         : {n_after_dedup:>10,}")
print(f"  rows dropped (dup IDs)   : {n_before_dedup - n_after_dedup:>10,}")
print(f"  rows in cleaned frame    : {n_rows_clean:>10,}")
print()

# --- 8. Sanity checks before writing ---------------------------------------
assert n_rows_clean == n_after_dedup, "row count mismatch after collect"
assert comments_clean.schema["body_normalized"] == pl.String
assert comments_clean.schema["rating_band"] == pl.String
assert comments_clean.schema["is_valid_rating_band"] == pl.Boolean

# Every rating_band must be one of {low, medium, high, invalid}
band_values = set(comments_clean["rating_band"].unique().to_list())
assert band_values.issubset({"low", "medium", "high", "invalid"}), band_values

# --- 9. Write parquet ------------------------------------------------------
comments_clean.write_parquet(
    COMMENTS_CLEAN_PARQUET,
    compression="zstd",
)
print(f"Wrote: {COMMENTS_CLEAN_PARQUET}")
print(f"Size : {COMMENTS_CLEAN_PARQUET.stat().st_size / 1024**2:,.1f} MB")
print(f"Rows : {n_rows_clean:,}")
print(f"Cols : {len(comments_clean.columns)} -> {comments_clean.columns}")

# --- 10. Post-write sanity: reread schema and a tiny slice ----------------
reread = pl.read_parquet(COMMENTS_CLEAN_PARQUET, n_rows=5)
print()
print("First 5 rows of the written parquet (body_normalized truncated to 80 chars):")
with pl.Config(fmt_str_lengths=80, tbl_width_chars=200):
    print(reread.select("id", "body_normalized", "recommendation_status",
                        "rate", "rating_band", "is_valid_rating_band"))

B2 Data Cleaning summary:
  rows before dedup        :  6,156,289
  rows after dedup         :  6,149,834
  rows dropped (dup IDs)   :      6,455
  rows in cleaned frame    :  6,149,834

Wrote: /home/jvd/Desktop/persian-nlp-pipeline/data/processed/comments_clean.parquet
Size : 178.5 MB
Rows : 6,149,834
Cols : 8 -> ['id', 'recommendation_status', 'rate', 'product_id', 'is_buyer', 'rating_band', 'is_valid_rating_band', 'body_normalized']

First 5 rows of the written parquet (body_normalized truncated to 80 chars):
shape: (5, 6)
┌──────────┬───────────────────────────────────────────────────────────────────────────────────┬───────────────────────┬──────┬─────────────┬──────────────────────┐
│ id       ┆ body_normalized                                                                   ┆ recommendation_status ┆ rate ┆ rating_band ┆ is_valid_rating_band │
│ ---      ┆ ---                                                                               ┆ ---                   ┆ ---  ┆ ---       

In [5]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 4 (temporary sanity check)
# ============================================================
# Quick verification that the parquet written in B2 matches
# expectations from Part A after cleaning. Not part of B3–B5.
# ============================================================

clean = pl.read_parquet(COMMENTS_CLEAN_PARQUET)

print("Parquet sanity check:")
print(f"  rows               : {clean.height:,}")
print(f"  cols               : {len(clean.columns)}")
print()

# 1. Missing body should equal the 637 sentinel rows (minus any removed by dedup)
n_body_null = clean.filter(pl.col("body_normalized").is_null()).height
print(f"  body_normalized null : {n_body_null:,}  (expected ~637)")

# 2. recommendation_status distribution after cleaning
print()
print("  recommendation_status (after cleaning):")
print(clean["recommendation_status"].value_counts().sort("count", descending=True))

# 3. rating_band distribution after cleaning
print()
print("  rating_band (after cleaning):")
print(clean["rating_band"].value_counts().sort("rating_band"))

# 4. is_valid_rating_band: only 1 row should be False (rate=2500)
n_invalid = clean.filter(~pl.col("is_valid_rating_band")).height
print()
print(f"  invalid rating_band rows : {n_invalid}  (expected 1)")

# 5. Product count (should be slightly lower than 331,548 due to dedup)
n_products = clean.select(pl.col("product_id").n_unique()).item()
print(f"  unique products    : {n_products:,}  (was 331,548 in A1 before dedup)")

Parquet sanity check:
  rows               : 6,149,834
  cols               : 8

  body_normalized null : 637  (expected ~637)

  recommendation_status (after cleaning):
shape: (4, 2)
┌───────────────────────┬─────────┐
│ recommendation_status ┆ count   │
│ ---                   ┆ ---     │
│ str                   ┆ u32     │
╞═══════════════════════╪═════════╡
│ recommended           ┆ 4158686 │
│ null                  ┆ 893723  │
│ no_idea               ┆ 593316  │
│ not_recommended       ┆ 504109  │
└───────────────────────┴─────────┘

  rating_band (after cleaning):
shape: (4, 2)
┌─────────────┬─────────┐
│ rating_band ┆ count   │
│ ---         ┆ ---     │
│ str         ┆ u32     │
╞═════════════╪═════════╡
│ high        ┆ 4000554 │
│ invalid     ┆ 1       │
│ low         ┆ 1091285 │
│ medium      ┆ 1057994 │
└─────────────┴─────────┘

  invalid rating_band rows : 1  (expected 1)
  unique products    : 331,512  (was 331,548 in A1 before dedup)


In [6]:
# ============================================================
# 02_preprocessing_and_split.ipynb — Cell 5: B3 Product-Grouped Split
# ============================================================
# Goal (per Definition):
#   - deterministic, product-grouped 70/15/15 split
#   - every product lands in exactly one split
#   - each split is reasonably representative of both targets
#
# No files are written. B5 writes the split CSVs.
# ============================================================

# --- 1. Load the cleaned parquet ------------------------------------------
clean = pl.read_parquet(COMMENTS_CLEAN_PARQUET)

print(f"Loaded parquet: {clean.height:,} rows, {len(clean.columns)} cols")
print()

# --- 2. Compute a per-product "majority class" for stratification -------
# For each product, we pick the most common non-null recommendation_status.
# Products with only null labels get assigned to a synthetic class "__none__"
# so they still participate in stratification (they are the natural-unlabeled
# candidates for the semi-supervised extension).
product_majority = (
    clean
    .group_by("product_id")
    .agg(
        pl.len().alias("n_reviews"),
        pl.col("recommendation_status")
          .drop_nulls()
          .mode()
          .first()
          .alias("majority_class"),
    )
    .with_columns(
        pl.col("majority_class").fill_null("__none__")
    )
)

print("Per-product majority class distribution:")
print(product_majority["majority_class"].value_counts().sort("count", descending=True))
print()

# --- 3. Stratified 70/15/15 split on products ----------------------------
# For each majority class, we independently split its products into three
# buckets using a seeded random permutation. This guarantees that the class
# proportions are (approximately) preserved across splits.
TRAIN_FRAC = 0.70
VAL_FRAC   = 0.15
TEST_FRAC  = 0.15

# seeded random column for reproducibility, applied within each class
product_majority = product_majority.with_columns(
    pl.int_range(pl.len()).shuffle(seed=SPLIT_SEED).over("majority_class").alias("rand")
)

# Within each class, compute the quantile thresholds for the split
product_majority = product_majority.with_columns(
    pl.len().over("majority_class").alias("class_size"),
).with_columns(
    (
        pl.col("rand")
        < (pl.col("class_size") * TRAIN_FRAC).floor().cast(pl.Int64)
    ).alias("in_train"),
    (
        (pl.col("rand") >= (pl.col("class_size") * TRAIN_FRAC).floor().cast(pl.Int64))
        & (pl.col("rand") < ((pl.col("class_size") * (TRAIN_FRAC + VAL_FRAC)).floor().cast(pl.Int64)))
    ).alias("in_val"),
    (
        pl.col("rand") >= ((pl.col("class_size") * (TRAIN_FRAC + VAL_FRAC)).floor().cast(pl.Int64))
    ).alias("in_test"),
)

# Assign a split label per product
product_split = (
    product_majority
    .select("product_id", "majority_class", "n_reviews", "rand", "in_train", "in_val", "in_test")
    .with_columns(
        pl.when(pl.col("in_train")).then(pl.lit("train"))
          .when(pl.col("in_val")).then(pl.lit("validation"))
          .when(pl.col("in_test")).then(pl.lit("test"))
          .otherwise(pl.lit("ERROR"))
          .alias("split")
    )
    .drop("in_train", "in_val", "in_test")
)

# --- 4. Verify every product lands in exactly one split -----------------
split_counts = product_split["split"].value_counts().sort("split")
print("Products per split:")
print(split_counts)
print()

n_products = product_split.height
n_train_p = split_counts.filter(pl.col("split") == "train")["count"][0]
n_val_p   = split_counts.filter(pl.col("split") == "validation")["count"][0]
n_test_p  = split_counts.filter(pl.col("split") == "test")["count"][0]

print(f"Total products       : {n_products:,}")
print(f"  train products     : {n_train_p:,}  ({n_train_p/n_products*100:.2f}%)")
print(f"  validation products: {n_val_p:,}  ({n_val_p/n_products*100:.2f}%)")
print(f"  test products      : {n_test_p:,}  ({n_test_p/n_products*100:.2f}%)")
print()

# Should be: every product exactly once
assert n_train_p + n_val_p + n_test_p == n_products, "Products not partitioned correctly"

# --- 5. Join split label back to rows -----------------------------------
rows_with_split = (
    clean
    .join(product_split.select("product_id", "split"), on="product_id", how="inner")
)

# Split into three DataFrames
train_df = rows_with_split.filter(pl.col("split") == "train").drop("split")
val_df   = rows_with_split.filter(pl.col("split") == "validation").drop("split")
test_df  = rows_with_split.filter(pl.col("split") == "test").drop("split")

print("Rows per split:")
print(f"  train      : {train_df.height:,}  ({train_df.height/rows_with_split.height*100:.2f}%)")
print(f"  validation : {val_df.height:,}    ({val_df.height/rows_with_split.height*100:.2f}%)")
print(f"  test       : {test_df.height:,}   ({test_df.height/rows_with_split.height*100:.2f}%)")
print(f"  total      : {rows_with_split.height:,}")
print()

# --- 6. Verify product-grouped split: no overlap ------------------------
train_products = set(train_df["product_id"].to_list())
val_products   = set(val_df["product_id"].to_list())
test_products  = set(test_df["product_id"].to_list())

assert train_products.isdisjoint(val_products),  "product_id in train AND validation!"
assert train_products.isdisjoint(test_products), "product_id in train AND test!"
assert val_products.isdisjoint(test_products),   "product_id in validation AND test!"

print("Product-grouping verified: no product appears in more than one split.")
print()

# --- 7. Representativeness check: recommendation_status ----------------
def rec_dist(df: pl.DataFrame) -> dict:
    d = df.filter(pl.col("recommendation_status").is_not_null())["recommendation_status"].value_counts()
    total = d["count"].sum()
    return {row["recommendation_status"]: row["count"]/total*100 for row in d.iter_rows(named=True)}

print("recommendation_status % per split (null labels excluded):")
rec_train = rec_dist(train_df)
rec_val   = rec_dist(val_df)
rec_test  = rec_dist(test_df)
for label in ["recommended", "not_recommended", "no_idea"]:
    print(f"  {label:<18}  train={rec_train.get(label,0):6.2f}%  val={rec_val.get(label,0):6.2f}%  test={rec_test.get(label,0):6.2f}%")
print()

# --- 8. Representativeness check: rating_band --------------------------
def band_dist(df: pl.DataFrame) -> dict:
    d = df.filter(pl.col("is_valid_rating_band"))["rating_band"].value_counts()
    total = d["count"].sum()
    return {row["rating_band"]: row["count"]/total*100 for row in d.iter_rows(named=True)}

print("rating_band % per split (invalid rows excluded):")
band_train = band_dist(train_df)
band_val   = band_dist(val_df)
band_test  = band_dist(test_df)
for label in ["low", "medium", "high"]:
    print(f"  {label:<18}  train={band_train.get(label,0):6.2f}%  val={band_val.get(label,0):6.2f}%  test={band_test.get(label,0):6.2f}%")
print()

# --- 9. Assess closeness of distributions ------------------------------
# Rule of thumb: if max spread across splits is <= 2 percentage points
# for every class, we consider the split "reasonably representative".
MAX_ALLOWED_SPREAD = 2.0

def max_spread(dist_dicts: list[dict], label: str) -> tuple[bool, float]:
    percentages = [d.get(label, 0) for d in dist_dicts]
    spread = max(percentages) - min(percentages)
    return (spread <= MAX_ALLOWED_SPREAD, spread)

print("Representativeness assessment:")
rec_ok = True
band_ok = True
for label in ["recommended", "not_recommended", "no_idea"]:
    ok, spread = max_spread([rec_train, rec_val, rec_test], label)
    print(f"  rec  {label:<18} spread = {spread:.2f} pp  ({'OK' if ok else 'WARN'})")
    rec_ok = rec_ok and ok
for label in ["low", "medium", "high"]:
    ok, spread = max_spread([band_train, band_val, band_test], label)
    print(f"  band {label:<18} spread = {spread:.2f} pp  ({'OK' if ok else 'WARN'})")
    band_ok = band_ok and ok

print()
if rec_ok and band_ok:
    print(f"✅ All classes within {MAX_ALLOWED_SPREAD:.1f} pp spread — split is representative.")
else:
    print(f"⚠️  Some classes exceed {MAX_ALLOWED_SPREAD:.1f} pp — investigate before proceeding.")

Loaded parquet: 6,149,834 rows, 8 cols

Per-product majority class distribution:
shape: (4, 2)
┌─────────────────┬────────┐
│ majority_class  ┆ count  │
│ ---             ┆ ---    │
│ str             ┆ u32    │
╞═════════════════╪════════╡
│ recommended     ┆ 258699 │
│ not_recommended ┆ 27513  │
│ no_idea         ┆ 22965  │
│ __none__        ┆ 22335  │
└─────────────────┴────────┘

Products per split:
shape: (3, 2)
┌────────────┬────────┐
│ split      ┆ count  │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ test       ┆ 49728  │
│ train      ┆ 232057 │
│ validation ┆ 49727  │
└────────────┴────────┘

Total products       : 331,512
  train products     : 232,057  (70.00%)
  validation products: 49,727  (15.00%)
  test products      : 49,728  (15.00%)

Rows per split:
  train      : 4,316,423  (70.19%)
  validation : 913,570    (14.86%)
  test       : 919,841   (14.96%)
  total      : 6,149,834

Product-grouping verified: no product appears in more than one s